# Fontainebleau benchmark notebook

This notebook accompanies the Fontainebleau example implemented in
`examples/natural/fontainebleau/`.

Like the refactored Holten example, the workflow is split into:
- `fontainebleau_case.py` for paths, YAML loading, and effective configs;
- `fontainebleau_benchmark.py` for lightweight pre-model summaries;
- `run_fontainebleau.py` for orchestration around `scripts/launcher.py`.

The notebook follows the same sequence: inspect the configured case,
optionally override the YAML, generate the local benchmark artifacts, then
run the standard single-date launcher in a notebook-friendly way.


In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
for parent in [ROOT, *ROOT.parents]:
    if (parent / "pyproject.toml").exists() and (parent / "pyage").exists():
        ROOT = parent
        break

default_results_root = Path.home() / "results" / "PyAge"
RESULTS_ROOT = Path(os.environ.get("PYAGE_RESULTS_DIR", str(default_results_root)))
os.environ["PYAGE_RESULTS_DIR"] = str(RESULTS_ROOT)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

EXAMPLE_DIR = ROOT / "examples" / "natural" / "fontainebleau"
PARAMS_PATH = EXAMPLE_DIR / "exemple_fontainebleau.yaml"
RUNNER_PATH = EXAMPLE_DIR / "run_fontainebleau.py"
CASE_HELPERS_PATH = EXAMPLE_DIR / "fontainebleau_case.py"
BENCHMARK_HELPERS_PATH = EXAMPLE_DIR / "fontainebleau_benchmark.py"

from examples.natural.fontainebleau.fontainebleau_case import build_context, build_effective_config, dump_yaml
from examples.natural.fontainebleau.fontainebleau_benchmark import prepare_fontainebleau_case

ctx = build_context(PARAMS_PATH)
prepared = prepare_fontainebleau_case(PARAMS_PATH)

print("CWD:", Path.cwd())
print("ROOT:", ROOT)
print("Example dir:", EXAMPLE_DIR)
print("Params:", PARAMS_PATH)
print("Runner:", RUNNER_PATH)
print("Results root:", RESULTS_ROOT)
print("Configured dataset:", ctx.params.dataset_name)
print("Configured LPM:", ctx.params.lpm_model_name)


In [ ]:
%matplotlib inline


## Reference files

The notebook keeps the Fontainebleau runner and YAML configuration visible,
and shows the currently selected observations prepared by the local helpers.


In [ ]:
import yaml

print("run_fontainebleau.py")
print("=" * 80)
print(RUNNER_PATH.read_text(encoding="utf-8"))

with PARAMS_PATH.open("r", encoding="utf-8") as handle:
    base_config = yaml.safe_load(handle) or {}

print("\nBase YAML")
print("=" * 80)
print(yaml.safe_dump(base_config, sort_keys=False))

prepared.selected_observations


## Optional overrides

Edit `overrides` if you want to switch dataset, LPM, or run settings while
keeping the same Fontainebleau workflow structure.


In [ ]:
import yaml

overrides = {
    # "dataset": {"name": "fontainebleau_IMR"},
    # "lpm": {"model_name": "dirac_double"},
    # "run": {
    #     "reachable_concentrations": True,
    #     "objective_function": True,
    #     "calibration_metropolis_hastings": True,
    #     "calibration_simplex": True,
    # },
    # "reachable_concentrations": {"nmodels": 2000},
    # "objective_function": {"nmodels": 4000},
    # "calibration_metropolis_hastings": {"nstep": 1000},
    # "calibration_simplex": {"init_multiples_n": 3, "fuq_n": 20},
}

effective_config = build_effective_config(PARAMS_PATH, overrides=overrides)
if overrides:
    EFFECTIVE_PARAMS_PATH = EXAMPLE_DIR / "generated" / "launcher_configs" / "fontainebleau_notebook_launcher.yaml"
    dump_yaml(EFFECTIVE_PARAMS_PATH, effective_config)
else:
    EFFECTIVE_PARAMS_PATH = PARAMS_PATH
prepared = prepare_fontainebleau_case(EFFECTIVE_PARAMS_PATH)

print("Effective params:", EFFECTIVE_PARAMS_PATH)
print("Configured dataset:", prepared.context.params.dataset_name)
print("Configured LPM:", prepared.context.params.lpm_model_name)
print(yaml.safe_dump(effective_config, sort_keys=False))


## Run benchmark

This cell mirrors `run_fontainebleau.py`: it first writes the local
Fontainebleau benchmark artifacts, then executes `scripts/launcher.py`
through `runpy` with notebook-friendly CLI arguments and measures wall time.


In [ ]:
import runpy
from time import perf_counter

import pyage.global_parameters as gp
from scripts.common.launcher_params import load_params
from scripts.common.launcher_paths import results_directory
from examples.natural.fontainebleau.fontainebleau_benchmark import build_pre_model_figures, write_benchmark_summary, write_prepared_tables

prepared = prepare_fontainebleau_case(EFFECTIVE_PARAMS_PATH)
BENCHMARK_DIR = prepared.context.paths.benchmark_dir
write_prepared_tables(prepared, BENCHMARK_DIR / "prepared")
build_pre_model_figures(prepared, BENCHMARK_DIR / "pre_model")
write_benchmark_summary(prepared, BENCHMARK_DIR)

resolved_params = load_params(ROOT, EFFECTIVE_PARAMS_PATH)
EXPECTED_RESULTS_DIR = Path(results_directory(gp, resolved_params.dataset_name))
script_path = ROOT / "scripts" / "launcher.py"
saved_argv = sys.argv[:]

print("Dataset:", resolved_params.dataset_name)
print("LPM model:", resolved_params.lpm_model_name)
print("Expected results directory:", EXPECTED_RESULTS_DIR)

start = perf_counter()
try:
    sys.argv = [str(script_path), str(EFFECTIVE_PARAMS_PATH), "--inline"]
    runpy.run_path(str(script_path), run_name="__main__")
finally:
    sys.argv = saved_argv

elapsed_s = perf_counter() - start
print(f"Elapsed wall time: {elapsed_s:.2f} s")


## Output check

Quick listing of the most recent files written to the benchmark directory and
to the expected launcher results directory.


In [ ]:
benchmark_files = [path for path in BENCHMARK_DIR.rglob("*") if path.is_file()]
recent_benchmark_files = sorted(benchmark_files, key=lambda path: path.stat().st_mtime, reverse=True)
result_files = [path for path in EXPECTED_RESULTS_DIR.rglob("*") if path.is_file()]
recent_result_files = sorted(result_files, key=lambda path: path.stat().st_mtime, reverse=True)

print("Benchmark directory:", BENCHMARK_DIR)
print("Benchmark file count:", len(benchmark_files))
print("Most recent benchmark files:")
for path in recent_benchmark_files[:10]:
    print(path.relative_to(BENCHMARK_DIR))

print("\nResults directory:", EXPECTED_RESULTS_DIR)
print("Results file count:", len(result_files))
print("Most recent results files:")
for path in recent_result_files[:20]:
    print(path.relative_to(EXPECTED_RESULTS_DIR))
